# Text Summarizer: any length, any style (Colab T4)

**Model:** `google/flan-t5-large` (instruction-following, runs in ~3 GB on a T4)

| Feature | How |
|---|---|
| Short / medium / detailed | 30–50 / 100–150 / 200–300 words, or set your own word count |
| Paragraph / bullets / key points / executive | `STYLE` option (executive = title, overview, key points, advantages, challenges, takeaway) |
| Any input length | Text is split into overlapping chunks, each chunk is summarized, then combined and re-summarized |
| Focus + reading level | e.g. focus on *"safety and business"*, or *simple* language |
| Quality checks | length, unsupported names/numbers, repetition, coverage, readability |
| Evaluation | ROUGE and side-by-side model comparison |

**No out-of-memory:** float32 weights, batch of 4 chunks, `no_grad`, GPU cache cleared after every batch.
Set **Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q evaluate rouge_score sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.0 MB/s eta 0:00:00


## 1. Settings and helpers

In [ ]:
import re, math, time, gc, os
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import Markdown, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "no GPU (Runtime > Change runtime type > T4)")

# ---------------- settings (safe for a free T4) ----------------
MODEL_NAME   = "google/flan-t5-large"   # use "google/flan-t5-base" if you want it faster / lighter
CHUNK_TOKENS = 400    # max source tokens per chunk
MAX_INPUT    = 640    # chunk + instruction must fit in this
BATCH_SIZE   = 4      # lower to 2 if you ever see an out-of-memory error
NUM_BEAMS    = 4
TOK_PER_WORD = 1.35   # rough T5 tokens per English word
LENGTHS = {"short": (30, 50), "medium": (100, 150), "detailed": (200, 300)}  # words

# ---------------- text helpers ----------------
def clean_text(t):
    t = re.sub(r"<[^>]+>", " ", t)          # html tags
    t = re.sub(r"https?://\S+", " ", t)     # urls
    return re.sub(r"\s+", " ", t).strip()   # whitespace / line breaks (case is kept on purpose)

def wc(t):
    return len(t.split())

def split_sentences(text, max_words=120):
    """Sentence split; any run-on piece longer than max_words is cut so chunks never overflow."""
    parts = re.split(r'(?<=[.!?])\s+(?=[A-Z0-9"“‘(\[])', text.strip())
    out = []
    for p in parts:
        w = p.split()
        for i in range(0, len(w), max_words):
            piece = " ".join(w[i:i + max_words])
            if piece:
                out.append(piece)
    return out

def trim_to_words(text, max_words):
    """Cut at a sentence boundary so the result stays within max_words."""
    if wc(text) <= max_words:
        return text
    out, n = [], 0
    for s in split_sentences(text):
        if n + wc(s) > max_words:
            break
        out.append(s); n += wc(s)
    return " ".join(out) if out else " ".join(text.split()[:max_words])

def dedupe(sents, thr=0.7):
    """Drop near-duplicate sentences (created by overlapping chunks)."""
    keep, seen = [], []
    for s in sents:
        ws = set(re.findall(r"\w+", s.lower()))
        if not ws or any(len(ws & o) / len(ws | o) > thr for o in seen):
            continue
        keep.append(s); seen.append(ws)
    return keep

def _stem_tokens(s):
    toks = re.findall(r"[a-z0-9]+", s.lower())
    return [re.sub(r"(ing|ed|es|s)$", "", w) if len(w) > 4 else w
            for w in toks if w not in ENGLISH_STOP_WORDS]

def _tfidf():
    return TfidfVectorizer(tokenizer=_stem_tokens, token_pattern=None, lowercase=False)

def top_sentences(sents, k):
    """Pick the k most central sentences (kept in original order)."""
    if len(sents) <= k:
        return sents
    try:
        sim = cosine_similarity(_tfidf().fit_transform(sents))
    except ValueError:
        return sents[:k]
    return [sents[i] for i in sorted(np.argsort(-sim.sum(1))[:k])]

def count_syllables(w):
    return max(1, len(re.findall(r"[aeiouy]+", w.lower())))

device: cuda | Tesla T4


## 2. Summarizer

In [ ]:
class Summarizer:
    def __init__(self, model_name=MODEL_NAME, instruct=None):
        self.instruct = ("flan" in model_name.lower()) if instruct is None else instruct
        self.tok = AutoTokenizer.from_pretrained(model_name)
        # float32 on purpose: T5 models give NaN/garbage in fp16 on a T4
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()

    # ---------- low level ----------
    def _count_tokens(self, texts):
        return [len(x) for x in self.tok(texts, add_special_tokens=False)["input_ids"]]

    @torch.no_grad()
    def _generate(self, prompts, min_tok, max_tok, beams=NUM_BEAMS):
        outs = []
        for i in range(0, len(prompts), BATCH_SIZE):          # small batches => no OOM
            enc = self.tok(prompts[i:i + BATCH_SIZE], return_tensors="pt", padding=True,
                           truncation=True, max_length=MAX_INPUT).to(device)
            ids = self.model.generate(**enc, min_new_tokens=min_tok, max_new_tokens=max_tok,
                                      num_beams=beams, no_repeat_ngram_size=3, early_stopping=True)
            outs += [o.strip() for o in self.tok.batch_decode(ids, skip_special_tokens=True)]
            del enc, ids
            if device == "cuda":
                torch.cuda.empty_cache()
        return outs

    def _prompt(self, chunk, words, focus=None, level="standard"):
        if not self.instruct:
            return "summarize: " + chunk
        p = (f"Summarize the following text in about {words} words. Keep the key facts, names, "
             f"dates and numbers. Do not add anything that is not in the text.")
        if focus:
            p += f" Focus on: {focus}."
        if level == "simple":
            p += " Use simple, easy-to-read language."
        return p + "\n\nText: " + chunk

    # ---------- long-document handling ----------
    def make_chunks(self, text, min_parts=1, overlap=True):
        sents = split_sentences(text)
        lens = self._count_tokens(sents)
        limit = min(CHUNK_TOKENS, max(120, math.ceil(1.1 * sum(lens) / max(min_parts, 1))))
        chunks, cur, cur_len = [], [], 0
        for s, l in zip(sents, lens):
            if cur and cur_len + l > limit:
                chunks.append(" ".join(c for c, _ in cur))
                last = cur[-1] if overlap and cur[-1][1] < 0.25 * limit else None  # 1-sentence overlap
                cur, cur_len = ([last], last[1]) if last else ([], 0)
            cur.append((s, l)); cur_len += l
        if cur:
            chunks.append(" ".join(c for c, _ in cur))
        return chunks

    def select_focus(self, chunks, focus):
        try:
            X = _tfidf().fit_transform(chunks + [focus])
            sims = cosine_similarity(X[-1], X[:-1]).ravel()
        except ValueError:
            return chunks
        if sims.max() == 0:
            return chunks
        keep = max(2, math.ceil(len(chunks) * 0.5))
        return [chunks[i] for i in sorted(np.argsort(-sims)[:keep])]

    def _summarize_chunks(self, chunks, budget, focus, level):
        lens = self._count_tokens(chunks)
        prompts = [self._prompt(c, budget, focus, level) for c in chunks]
        lo = max(8, int(min(0.7 * budget * TOK_PER_WORD, 0.5 * min(lens))))
        hi = max(lo + 8, int(min(1.4 * budget * TOK_PER_WORD, max(lens))))
        return self._generate(prompts, lo, hi)

    def condense(self, text, target, min_parts=1, focus=None, level="standard"):
        """Map-reduce: summarize every chunk, join, and repeat until it fits the target."""
        cur, depth = text, 0
        while True:
            chunks = self.make_chunks(cur, min_parts if depth == 0 else 1, overlap=(depth == 0))
            if depth == 0 and focus and len(chunks) > 3:
                chunks = self.select_focus(chunks, focus)
            n = len(chunks)
            budget = target if n == 1 else int(min(110, max(20, 1.15 * target / n)))
            outs = self._summarize_chunks(chunks, budget, focus, level)
            combined = " ".join(dedupe(split_sentences(" ".join(outs))))
            depth += 1
            if n == 1 or wc(combined) <= 1.3 * target or depth >= 4 or wc(combined) >= 0.9 * wc(cur):
                return combined
            cur = combined

    def _paragraph(self, text, target, hi, level, focus):
        n_words = wc(text)
        min_parts = max(1, min(math.ceil(target / 70), n_words // 90))  # spread detail across the article
        return trim_to_words(self.condense(text, target, min_parts, focus, level), hi)

    # ---------- structured / executive summary ----------
    def extract_list(self, text, what, limit=5):
        chunks = self.make_chunks(text, 1, overlap=False)
        prompts = [f"List the {what} mentioned in the text, separated by semicolons. "
                   f"If none are mentioned, answer: none.\n\nText: {c}" for c in chunks]
        items = []
        for o in self._generate(prompts, 0, 60, beams=2):
            if re.match(r"^(none|no\b|not\b|n/a)", o.strip().lower()):
                continue
            items += [x.strip(" .") for x in re.split(r";|\n", o) if x.strip(" .")]
        items = dedupe([i[0].upper() + i[1:] for i in items])
        return items[:limit]

    def executive(self, text, level="standard", focus=None):
        ctx = self._paragraph(text, 200, 260, level, focus)
        overview = self._generate([self._prompt(ctx, 60, focus, level)], 40, 110)[0]
        points = top_sentences(split_sentences(ctx), 5)
        md = []
        if self.instruct:
            title = self._generate([f"Write a short title (at most 8 words) for this text.\n\nText: {ctx}"], 3, 24)[0]
            md.append(f"## {title.strip(' .')}\n")
        md.append(f"**Overview**\n\n{overview}\n")
        md.append("### Key points\n" + "\n".join(f"- {p}" for p in points) + "\n")
        if self.instruct:
            for head, what in [("Advantages", "benefits or advantages"),
                               ("Challenges", "risks, challenges or limitations")]:
                items = self.extract_list(text, what)
                if items:
                    md.append(f"### {head}\n" + "\n".join(f"- {i}" for i in items) + "\n")
            take = self._generate(["What is the main conclusion or takeaway of this text? "
                                   f"Answer in one sentence.\n\nText: {ctx}"], 8, 60)[0]
        else:
            take = split_sentences(ctx)[-1]
        md.append(f"### Main takeaway\n\n{take}")
        return "\n".join(md)

    # ---------- public API ----------
    def summarize(self, text, length="medium", style="paragraph", level="standard",
                  focus=None, words=None, checks=False):
        """length: short|medium|detailed   style: paragraph|bullets|key_points|executive
           level: standard|simple          focus: e.g. 'safety and business'   words: custom target"""
        text = clean_text(text)
        n_words = wc(text)
        if n_words < 40:
            return text  # nothing to shorten
        if words:
            target, (lo, hi) = int(words), (int(words * 0.85), int(words * 1.15))
        else:
            lo, hi = LENGTHS[length]; target = (lo + hi) // 2
        cap = max(20, int(0.6 * n_words))
        if target > cap:                                   # source shorter than requested summary
            target, lo, hi = cap, int(cap * 0.8), int(cap * 1.2)

        if style == "executive":
            out = self.executive(text, level, focus)
            if checks:
                body = "\n".join(l for l in out.splitlines() if not l.startswith(("#", "**")))
                print(self.check(text, body))
            return out

        summary = self._paragraph(text, target, hi, level, focus)
        sents = split_sentences(summary)
        k = max(3, round(target / 35))
        if style == "bullets":
            out = "\n".join(f"- {s}" for s in sents)
        elif style == "key_points":
            out = "\n".join(f"- {s}" for s in top_sentences(sents, k))
        else:
            out = summary
        if checks:
            print(self.check(text, summary, (lo, hi)))
        return out

    # ---------- quality checks (fast heuristics, no extra model, no extra GPU memory) ----------
    def check(self, source, summary, bounds=None):
        source = clean_text(source)
        summary = re.sub(r"^[-#*\s]+", "", summary, flags=re.M)      # strip markdown marks
        res = {"summary_words": wc(summary), "source_words": wc(source)}
        if bounds:
            res["requested_range"] = f"{bounds[0]}-{bounds[1]}"
            res["length_ok"] = bounds[0] * 0.9 <= wc(summary) <= bounds[1] * 1.1

        # factual consistency: names / numbers in the summary that never appear in the source
        src_words = set(re.findall(r"[a-z0-9]+", source.lower()))
        unsupported = set()
        for s in split_sentences(summary):
            first = (s.split() or [""])[0].strip('"“(')
            for t in re.findall(r"\b(?:[A-Z][A-Za-z0-9\-]*|\d[\d,.%]*)\b", s):
                if t == first and t.isalpha():
                    continue
                if any(p not in src_words for p in re.findall(r"[a-z0-9]+", t.lower())):
                    unsupported.add(t)
        res["unsupported_terms"] = sorted(unsupported) or "none"

        # repetition
        w = re.findall(r"\w+", summary.lower())
        tri = list(zip(w, w[1:], w[2:]))
        res["repetition"] = round(1 - len(set(tri)) / len(tri), 3) if tri else 0.0

        # coverage: how many equal slices of the source are reflected in the summary
        sents = split_sentences(source)
        if len(sents) >= 8:
            size = math.ceil(len(sents) / 8)
            parts = [" ".join(sents[i:i + size]) for i in range(0, len(sents), size)]
            try:
                X = _tfidf().fit_transform(parts + [summary])
                sims = cosine_similarity(X[-1], X[:-1]).ravel()
                res["coverage"] = f"{int((sims >= 0.08).sum())}/{len(parts)} sections"
            except ValueError:
                pass

        # readability (Flesch reading ease: higher = easier)
        sw = summary.split()
        ns = max(1, len(split_sentences(summary)))
        if sw:
            syl = sum(count_syllables(x) for x in sw)
            res["readability"] = round(206.835 - 1.015 * len(sw) / ns - 84.6 * syl / len(sw), 1)
        return res

## 3. Load the model

In [ ]:
summarizer = Summarizer(MODEL_NAME)
print("loaded:", MODEL_NAME)
if device == "cuda":
    print(f"GPU memory used: {torch.cuda.memory_allocated() / 1e9:.1f} GB")

config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

loaded: google/flan-t5-large
GPU memory used: 3.3 GB


## 4. Paste your text
Replace the sample article with any English text of any length.

In [ ]:
TEXT = """
Artificial intelligence has become one of the most influential technologies of the modern era, changing how people communicate, work, learn, and solve problems. Among the companies contributing to this transformation is Anthropic, an artificial intelligence research and development company known for creating Claude, a family of large language models designed to assist people with a wide range of tasks. Anthropic places particular emphasis on building AI systems that are useful, reliable, understandable, and safe. Its work reflects a broader effort to develop advanced AI capabilities while studying the risks that may arise as these systems become more powerful.

Anthropic was founded in 2021 by former OpenAI employees, including siblings Dario Amodei and Daniela Amodei. The company was established with the goal of advancing AI research while placing a strong emphasis on safety and responsible development. Its researchers and engineers investigate how AI models learn, how their behavior can be guided, and how potential risks can be reduced. Rather than treating safety as a final step after a model has been developed, Anthropic has presented it as an important part of the entire research and deployment process. This approach has helped establish the company's identity in the competitive artificial intelligence industry.

Claude is Anthropic's primary AI assistant and is designed to understand and generate human language. Depending on the model and product features available, Claude can help users write articles, summarize lengthy documents, explain complex subjects, analyze information, translate text, generate computer code, and solve problems that require several steps of reasoning. Users can interact with Claude through conversational prompts, making it useful for both everyday activities and specialized professional tasks. For example, a student can ask Claude to explain a difficult scientific concept in simple language, while a business analyst can use it to extract important findings from a lengthy report. A software developer can ask Claude to examine code, identify possible errors, suggest improvements, or help implement a new feature.

One important feature of the Claude family is that it includes models designed for different needs. Some models prioritize speed and lower costs, making them suitable for routine tasks and applications that process many requests. Other models are designed for more demanding activities, such as complex reasoning, software development, research, and long-running workflows. This variety allows developers and businesses to choose a model based on the complexity of a task, the response time they require, and their available budget. The exact capabilities, model names, and availability change as Anthropic introduces new versions.

Claude has become particularly relevant in software development. AI-assisted coding tools can help programmers understand unfamiliar codebases, generate functions, write tests, identify bugs, and explain technical concepts. Anthropic's Claude Code product extends these capabilities by helping developers work with code and development tools in their environments. Such systems may reduce the time spent on repetitive programming activities and allow engineers to focus more attention on system design, testing, and problem-solving. However, AI-generated code still requires careful review because it can contain security vulnerabilities, logical errors, inefficient implementations, or incorrect assumptions. Human expertise remains important when evaluating whether generated code is reliable and appropriate for a particular application.

Beyond programming, Claude can support businesses in areas such as customer service, document processing, research, content creation, and internal knowledge management. Organizations may use AI assistants to summarize meeting notes, draft reports, classify documents, search internal information, and automate parts of administrative workflows. When connected to suitable tools and authorized data sources, AI systems can also assist with more complex processes that involve multiple steps. These capabilities may improve productivity and help employees spend less time on repetitive tasks. Nevertheless, businesses must carefully manage access permissions, protect confidential information, verify important outputs, and ensure that automated systems follow organizational policies.

Anthropic's approach to AI safety is an important part of its public identity. One technique associated with the company is Constitutional AI, an approach that uses a set of principles to help guide model behavior. During training, models are encouraged to evaluate and revise responses according to specified principles, alongside other training and evaluation methods. The objective is to develop systems that are helpful while reducing harmful or inappropriate outputs. This approach is intended to make AI behavior more consistent and easier to guide, although it does not guarantee that a model will always behave correctly. AI systems can still misunderstand instructions, generate inaccurate information, or respond in unexpected ways.

Another important area of research is interpretability, which investigates how neural networks represent information and arrive at particular outputs. Modern language models contain large numbers of interacting computational components, making their internal behavior difficult to understand completely. Researchers study these systems to identify patterns, examine how particular concepts are represented, and investigate the mechanisms behind model decisions. Better interpretability could help scientists detect certain problems, understand model capabilities, and improve safety evaluations. However, explaining the internal behavior of advanced AI systems remains a challenging research problem, and a fluent explanation generated by a model should not automatically be treated as proof of how its internal computation actually worked.

Despite their capabilities, Claude and other AI assistants have several limitations. They can produce hallucinations, meaning statements that sound convincing but are inaccurate or unsupported. They may misunderstand ambiguous questions, overlook important context, or provide outdated information when they lack access to current sources. Their performance also depends on the quality of the input, the nature of the task, and the information available to them. Users should therefore verify important claims, particularly in fields such as medicine, law, finance, engineering, and scientific research. AI assistance can improve efficiency, but it should not replace qualified professional judgment when decisions carry significant consequences.

Privacy and data security are additional considerations. People may enter personal details, business records, research documents, or other sensitive information into AI applications. Organizations providing these systems must establish appropriate safeguards, explain their data practices, and control access to confidential material. Users should also understand the policies of the specific service they use, including how their information may be stored or processed. Businesses adopting Claude through an API or enterprise platform may have different contractual and data-handling arrangements from individuals using a consumer application. Careful evaluation of these differences is essential before introducing AI into sensitive workflows.

The expansion of AI also raises questions about employment and the future of work. Automation may reduce the need for some repetitive activities, while increasing demand for skills related to AI integration, data management, cybersecurity, critical thinking, and system evaluation. The overall impact will vary across industries and occupations. Some workers may benefit from AI tools that help them complete tasks more efficiently, while others may need training to adapt to changing job requirements. Educational institutions, businesses, and governments can help people prepare for these changes by supporting continuous learning and making opportunities for skill development more accessible.

Anthropic operates in a highly competitive market that includes companies such as OpenAI, Google, and other AI developers. These organizations compete on model capabilities, reliability, speed, pricing, developer tools, and enterprise services. Competition can encourage rapid innovation and expand access to useful technologies, but it also creates pressure to release increasingly capable systems. This makes independent evaluation, transparent reporting, appropriate safeguards, and responsible deployment important considerations. Companies must balance the desire to deliver new features with the need to understand the potential consequences of their products.

Looking ahead, Claude and similar AI systems may become more deeply integrated into education, scientific research, healthcare administration, software engineering, and everyday business operations. Improvements in reasoning, multimodal understanding, and tool use could allow these systems to support increasingly complex workflows. At the same time, greater capabilities may introduce new challenges involving cybersecurity, misinformation, privacy, reliability, and human oversight. Progress will depend not only on improving model performance but also on designing effective evaluation methods, clear accountability structures, and safeguards that work in real-world environments.

In conclusion, Anthropic and its Claude AI models represent an important part of the ongoing development of artificial intelligence. Claude offers practical assistance with writing, analysis, coding, research, and many other activities, while Anthropic emphasizes safety research and the responsible development of advanced AI systems. These technologies provide substantial opportunities to improve productivity and expand access to information, but they also have limitations that require careful attention. The long-term value of AI will depend on how effectively developers, organizations, governments, and individual users combine technological innovation with accuracy, privacy, accountability, and human judgment.
"""

# Or load a .txt file instead (uncomment):
# from google.colab import files
# up = files.upload()
# TEXT = open(list(up.keys())[0], encoding="utf-8").read()

## 5. Summarize
Pick the options in the form, then run.

In [ ]:
#@title Choose options and run
LENGTH = "medium"      #@param ["short", "medium", "detailed"]
STYLE = "bullets"      #@param ["paragraph", "bullets", "key_points", "executive"]
READING_LEVEL = "standard"  #@param ["standard", "simple"]
CUSTOM_WORDS = 0         #@param {type:"integer"}
FOCUS = ""               #@param {type:"string"}
SHOW_CHECKS = True       #@param {type:"boolean"}

t0 = time.time()
result = summarizer.summarize(
    TEXT,
    length=LENGTH,
    style=STYLE,
    level=READING_LEVEL,
    focus=FOCUS.strip() or None,
    words=CUSTOM_WORDS or None,
    checks=SHOW_CHECKS,
)
display(Markdown(result))
print(f"\n({wc(clean_text(TEXT))} words in -> {wc(result)} words out, {time.time() - t0:.0f}s)")

{'summary_words': 133, 'source_words': 1398, 'requested_range': '100-150', 'length_ok': True, 'unsupported_terms': 'none', 'repetition': 0.023, 'coverage': '8/8 sections', 'readability': 20.7}


- Anthropic is an artificial intelligence research and development company known for creating Claude, a family of large language models designed to assist people with a wide range of tasks.
- Anthropic's Claude family of artificial intelligence assistants can be used for a variety of tasks.
- Claude can help programmers understand unfamiliar codebases, generate functions, write tests, identify bugs, and explain technical concepts.
- Anthropic's approach to AI safety is an important part of its public identity.
- The objective is to develop systems that are helpful while reducing harmful or inappropriate outputs.
- Users should be aware of the limitations of AI assistants.
- They should also consider privacy and data security.
- The future of work will depend on AI tools.
- Anthropic and its Claude AI models represent an important part of the ongoing development of artificial intelligence.


(1398 words in -> 142 words out, 9s)


In [ ]:
# Quick look at every length (paragraph style)
for ln in ["short", "medium", "detailed"]:
    out = summarizer.summarize(TEXT, length=ln)
    print(f"\n=== {ln.upper()} ({wc(out)} words) ===\n{out}")


=== SHORT (39 words) ===
Anthropic is an artificial intelligence research and development company known for creating Claude, a family of large language models designed to assist people with a wide range of tasks. Users should be aware of the limitations of AI assistants.

=== MEDIUM (133 words) ===
Anthropic is an artificial intelligence research and development company known for creating Claude, a family of large language models designed to assist people with a wide range of tasks. Anthropic's Claude family of artificial intelligence assistants can be used for a variety of tasks. Claude can help programmers understand unfamiliar codebases, generate functions, write tests, identify bugs, and explain technical concepts. Anthropic's approach to AI safety is an important part of its public identity. The objective is to develop systems that are helpful while reducing harmful or inappropriate outputs. Users should be aware of the limitations of AI assistants. They should also consider priv

## 6. Evaluate (ROUGE) and compare models
Needs matching `texts` and `references` lists. The demo uses `samsum-validation.csv` if you upload it.

In [ ]:
def evaluate_rouge(summarizer, texts, references, length="short", style="paragraph"):
    """texts and references must be matching lists (article i <-> reference summary i)."""
    import evaluate
    rouge = evaluate.load("rouge")
    preds = [summarizer.summarize(t, length=length, style=style) for t in texts]
    return rouge.compute(predictions=preds, references=references)

def compare_models(model_names, texts, references, length="short"):
    """Run the same articles through several models, one at a time, to avoid out-of-memory."""
    import evaluate, pandas as pd
    rouge, rows = evaluate.load("rouge"), []
    for name in model_names:
        s, t0 = Summarizer(name), time.time()
        preds = [s.summarize(t, length=length) for t in texts]
        r = rouge.compute(predictions=preds, references=references)
        rows.append({"model": name, **{k: round(float(v), 4) for k, v in r.items()},
                     "avg_words": round(np.mean([wc(p) for p in preds]), 1),
                     "sec": round(time.time() - t0)})
        del s; gc.collect()
        if device == "cuda":
            torch.cuda.empty_cache()
    return pd.DataFrame(rows)

# ---- demo on the SAMSum validation file (skipped if the CSV is not in the Colab files) ----
if os.path.exists("samsum-validation.csv"):
    import pandas as pd
    val = pd.read_csv("samsum-validation.csv").sample(n=20, random_state=42)
    texts, refs = val["dialogue"].tolist(), val["summary"].tolist()
    print(evaluate_rouge(summarizer, texts, refs))
    # Compare models (each is loaded and freed in turn):
    # print(compare_models(["t5-small", "google/flan-t5-base", "google/flan-t5-large"], texts, refs))
else:
    print("Upload samsum-validation.csv (or build your own `texts` / `refs` lists) to run ROUGE.")

{'rouge1': np.float64(0.42760037767820536), 'rouge2': np.float64(0.21081020961162084), 'rougeL': np.float64(0.3432225052684551), 'rougeLsum': np.float64(0.3430850632673278)}


## 7. Optional: fine-tune on your own data
Off by default. Not needed for articles, only for dialogue-style summaries like SAMSum. Uses `flan-t5-base` with Adafactor and gradient checkpointing, so it fits on a T4.

In [ ]:
FINETUNE = True  #@param {type:"boolean"}

if FINETUNE:
    import pandas as pd
    from datasets import Dataset
    from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

    del summarizer; gc.collect(); torch.cuda.empty_cache()      # free GPU memory first

    base = "google/flan-t5-base"
    tok = AutoTokenizer.from_pretrained(base)
    ft_model = AutoModelForSeq2SeqLM.from_pretrained(base)
    ft_model.config.use_cache = False                            # needed for gradient checkpointing

    cols = ["dialogue", "summary"]
    train_df = pd.read_csv("samsum-train.csv").sample(n=4000, random_state=42)[cols].reset_index(drop=True)
    val_df = pd.read_csv("samsum-validation.csv").sample(n=500, random_state=42)[cols].reset_index(drop=True)

    def prep(batch):
        x = tok(["summarize: " + clean_text(d) for d in batch["dialogue"]], max_length=512, truncation=True)
        x["labels"] = tok(text_target=[clean_text(s) for s in batch["summary"]],
                          max_length=128, truncation=True)["input_ids"]
        return x

    train_ds = Dataset.from_pandas(train_df).map(prep, batched=True, remove_columns=cols)
    val_ds = Dataset.from_pandas(val_df).map(prep, batched=True, remove_columns=cols)

    args = Seq2SeqTrainingArguments(
        output_dir="./results",
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,        # effective batch = 16
        per_device_eval_batch_size=4,
        learning_rate=3e-4,
        optim="adafactor",                    # far less memory than AdamW
        gradient_checkpointing=True,          # trades a bit of speed for a lot of memory
        fp16=False,                           # T5 is unstable in fp16
        weight_decay=0.01,
        warmup_steps=50,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        report_to="none",
    )
    trainer = Seq2SeqTrainer(model=ft_model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                             data_collator=DataCollatorForSeq2Seq(tok, model=ft_model, label_pad_token_id=-100))
    trainer.train()

    ft_model.config.use_cache = True
    ft_model.save_pretrained("./summary_model"); tok.save_pretrained("./summary_model")
    del trainer, ft_model; gc.collect(); torch.cuda.empty_cache()

    # Use it (dialogue-style summaries). instruct=False because it was trained with the "summarize:" prefix.
    summarizer = Summarizer("./summary_model", instruct=False)
    print(summarizer.summarize("Amy: hey are we still on for lunch? Bob: yes, 1pm at the usual place. "
                               "Amy: great, I'll bring the documents. Bob: perfect, see you there.", length="short"))

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

...

Epoch,Training Loss,Validation Loss
1,6.035165,1.421351
2,4.961460,1.424129
3,4.303488,1.453498


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Amy: hey are we still on for lunch? Bob: yes, 1pm at the usual place. Amy: great, I'll bring the documents. Bob: perfect, see you there.
